In [1]:

# ============================================================
# STATISTICAL FORECASTING MODELS
# ============================================================

import os
import json
import warnings
import joblib
import numpy as np
import pandas as pd

from itertools import product

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import (
    SimpleExpSmoothing,
    Holt,
    ExponentialSmoothing
)
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Project directory and output folders
# ------------------------------------------------------------

cwd = os.getcwd()

PROJECT_ROOT = (
    os.path.dirname(cwd)
    if os.path.basename(cwd).lower() == "notebooks"
    else cwd
)

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs")
MODEL_DIR = os.path.join(PROJECT_ROOT, "models", "statistical")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# ------------------------------------------------------------
# Load forecasting dataset
# ------------------------------------------------------------

source_file = os.path.join(
    OUTPUT_DIR,
    "forecasting_framework_dataset.csv"
)

if not os.path.exists(source_file):
    raise FileNotFoundError(
        "forecasting_framework_dataset.csv not found. "
        "Run the forecasting framework notebook first."
    )

df = pd.read_csv(source_file)

df.columns = (
    df.columns.astype(str).str.strip().str.lower()
)

# ------------------------------------------------------------
# Identify forecasting target and date column
# ------------------------------------------------------------

date_candidates = [
    "date",
    "report_date",
    "observation_date",
    "snapshot_date"
]

date_col = next(
    (c for c in date_candidates if c in df.columns),
    None
)

if date_col is None:
    raise ValueError("Date column not found.")

target_candidates = [
    "children_in_hhs_care",
    "total_children",
    "total_care_load",
    "care_load",
    "placement_demand"
]

target_col = next(
    (c for c in target_candidates if c in df.columns),
    None
)

if target_col is None:
    raise ValueError(
        "Forecasting target not found. "
        "Check the actual target column in your dataset."
    )

df[date_col] = pd.to_datetime(
    df[date_col],
    errors="coerce"
)

df[target_col] = pd.to_numeric(
    df[target_col],
    errors="coerce"
)

df = (
    df.dropna(subset=[date_col, target_col])
    .sort_values(date_col)
    .drop_duplicates(subset=[date_col])
    .reset_index(drop=True)
)

print("Dataset:", source_file)
print("Date column:", date_col)
print("Target:", target_col)
print("Observations:", len(df))

# ------------------------------------------------------------
# Load chronological data boundaries
# ------------------------------------------------------------

split_file = os.path.join(
    OUTPUT_DIR,
    "chronological_split_summary.csv"
)

if not os.path.exists(split_file):
    raise FileNotFoundError(
        "chronological_split_summary.csv not found."
    )

split_summary = pd.read_csv(split_file)

def get_split_date(dataset_name, column_name):

    result = split_summary.loc[
        split_summary["Dataset"] == dataset_name,
        column_name
    ]

    if result.empty:
        raise ValueError(
            f"{dataset_name} boundary not found."
        )

    return pd.to_datetime(result.iloc[0])


train_end = get_split_date("Training", "End_Date")

validation_end = get_split_date("Validation", "End_Date")

train_data = df[
    df[date_col] <= train_end
].copy()

validation_data = df[
    (df[date_col] > train_end)
    & (df[date_col] <= validation_end)
].copy()

test_data = df[
    df[date_col] > validation_end
].copy()

if min(
    len(train_data),
    len(validation_data),
    len(test_data)
) == 0:
    raise ValueError("One or more chronological partitions are empty.")

print("\nChronological split:")
print("Training:", len(train_data))
print("Validation:", len(validation_data))
print("Testing:", len(test_data))

# ------------------------------------------------------------
# Define forecast horizons
# ------------------------------------------------------------

horizons = {
    "1_day": 1,
    "7_day": 7,
    "14_day": 14,
    "30_day": 30
}

config_file = os.path.join(
    OUTPUT_DIR,
    "forecasting_configuration.json"
)

if os.path.exists(config_file):

    with open(config_file, "r") as file:
        phase8_config = json.load(file)

    saved_steps = phase8_config.get(
        "forecast_horizon_steps", {}
    )

    for horizon_name in horizons:

        if horizon_name in saved_steps:
            horizons[horizon_name] = int(
                saved_steps[horizon_name]
            )

print("\nForecast horizons:", horizons)

# ------------------------------------------------------------
# Prepare chronological target series
# ------------------------------------------------------------

y_train = train_data[target_col].astype(float)

y_validation = validation_data[target_col].astype(float)

y_test = test_data[target_col].astype(float)

if len(y_train) < 30:
    raise ValueError(
        "Insufficient training observations for statistical models."
    )

max_horizon = max(horizons.values())

# ------------------------------------------------------------
# Determine differencing requirements
# ------------------------------------------------------------

adf_result = adfuller(
    y_train.dropna(),
    autolag="AIC"
)

adf_statistic = adf_result[0]
adf_pvalue = adf_result[1]

differencing_order = (
    0 if adf_pvalue < 0.05 else 1
)

print("\nStationarity test")
print("ADF statistic:", adf_statistic)
print("ADF p-value:", adf_pvalue)
print("Selected differencing:", differencing_order)

# ------------------------------------------------------------
# Identify candidate ARIMA parameters
# ------------------------------------------------------------

arima_candidates = []

best_arima_aic = np.inf
best_arima_order = None

for p, q in product(range(3), range(3)):

    order = (p, differencing_order, q)

    try:

        fitted = ARIMA(
            y_train,
            order=order,
            enforce_stationarity=False,
            enforce_invertibility=False
        ).fit()

        arima_candidates.append({
            "Order": str(order),
            "AIC": fitted.aic,
            "BIC": fitted.bic
        })

        if fitted.aic < best_arima_aic:
            best_arima_aic = fitted.aic
            best_arima_order = order

    except Exception:
        continue

if best_arima_order is None:
    raise RuntimeError("ARIMA model fitting failed.")

arima_candidates_df = pd.DataFrame(
    arima_candidates
).sort_values("AIC")

print("\nSelected ARIMA order:", best_arima_order)

# ------------------------------------------------------------
# Identify seasonal periods
# ------------------------------------------------------------

median_interval = df[date_col].diff().median()

if median_interval <= pd.Timedelta(days=1.5):

    reporting_frequency = "Daily"
    seasonal_period = 7

elif median_interval <= pd.Timedelta(days=8):

    reporting_frequency = "Weekly"
    seasonal_period = 52

elif median_interval <= pd.Timedelta(days=35):

    reporting_frequency = "Monthly"
    seasonal_period = 12

else:

    reporting_frequency = "Irregular_or_Lower_Frequency"
    seasonal_period = None

if (
    seasonal_period is not None
    and len(y_train) < seasonal_period * 2
):
    seasonal_period = None

print("\nReporting frequency:", reporting_frequency)
print("Seasonal period:", seasonal_period)

# ------------------------------------------------------------
# Identify candidate SARIMA parameters
# ------------------------------------------------------------

best_sarima_order = None
best_seasonal_order = None
best_sarima_aic = np.inf

sarima_candidates = []

if seasonal_period is not None:

    for p, q, P, Q, D in product(
        range(2),
        range(2),
        range(2),
        range(2),
        range(2)
    ):

        order = (p, differencing_order, q)

        seasonal_order = (
            P, D, Q, seasonal_period
        )

        try:

            fitted = SARIMAX(
                y_train,
                order=order,
                seasonal_order=seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False
            ).fit(
                disp=False,
                maxiter=100
            )

            sarima_candidates.append({
                "Order": str(order),
                "Seasonal_Order": str(seasonal_order),
                "AIC": fitted.aic,
                "BIC": fitted.bic
            })

            if fitted.aic < best_sarima_aic:

                best_sarima_aic = fitted.aic
                best_sarima_order = order
                best_seasonal_order = seasonal_order

        except Exception:
            continue

sarima_candidates_df = pd.DataFrame(
    sarima_candidates
)

print("\nSelected SARIMA order:", best_sarima_order)
print("Selected seasonal order:", best_seasonal_order)

# ------------------------------------------------------------
# Train statistical forecasting models
# ------------------------------------------------------------

def train_statistical_models(series):

    fitted_models = {}

    fitted_models["ARIMA"] = ARIMA(
        series,
        order=best_arima_order,
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit()

    if best_sarima_order is not None:

        try:

            fitted_models["SARIMA"] = SARIMAX(
                series,
                order=best_sarima_order,
                seasonal_order=best_seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False
            ).fit(
                disp=False,
                maxiter=200
            )

        except Exception as error:
            print("SARIMA fitting skipped:", error)

    fitted_models["Simple_Exponential_Smoothing"] = (
        SimpleExpSmoothing(
            series,
            initialization_method="estimated"
        ).fit(optimized=True)
    )

    fitted_models["Holt_Linear_Trend"] = (
        Holt(
            series,
            initialization_method="estimated"
        ).fit(optimized=True)
    )

    if (
        seasonal_period is not None
        and len(series) >= seasonal_period * 2
    ):

        try:

            fitted_models["Holt_Winters"] = (
                ExponentialSmoothing(
                    series,
                    trend="add",
                    seasonal="add",
                    seasonal_periods=seasonal_period,
                    initialization_method="estimated"
                ).fit(optimized=True)
            )

        except Exception as error:
            print("Holt-Winters fitting skipped:", error)

    return fitted_models


validation_models = train_statistical_models(y_train)

print("\nTrained validation models:")

for name in validation_models:
    print("-", name)

# ------------------------------------------------------------
# Generate validation forecasts
# ------------------------------------------------------------

validation_forecasts = []

for model_name, fitted in validation_models.items():

    try:

        forecasts = np.asarray(
            fitted.forecast(steps=max_horizon)
        ).flatten()

        for horizon_name, steps in horizons.items():

            if steps > len(y_validation):
                continue

            if steps > len(forecasts):
                continue

            actual = y_validation.iloc[steps - 1]

            predicted = forecasts[steps - 1]

            actual_date = validation_data.iloc[
                steps - 1
            ][date_col]

            if not np.isfinite(predicted):
                continue

            validation_forecasts.append({

                "Dataset": "Validation",
                "Model": model_name,
                "Horizon": horizon_name,
                "Forecast_Origin": train_data[
                    date_col
                ].iloc[-1],
                "Actual_Target_Date": actual_date,
                "Actual": float(actual),
                "Predicted": float(predicted)

            })

    except Exception as error:
        print(f"{model_name} validation error:", error)

validation_forecasts = pd.DataFrame(
    validation_forecasts
)

if validation_forecasts.empty:
    raise RuntimeError("No validation forecasts generated.")

# ------------------------------------------------------------
# Refit models on training and validation observations
# ------------------------------------------------------------

development_data = pd.concat(
    [train_data, validation_data],
    ignore_index=True
)

y_development = development_data[
    target_col
].astype(float)

test_models = train_statistical_models(
    y_development
)

# ------------------------------------------------------------
# Generate final testing forecasts
# ------------------------------------------------------------

test_forecasts = []

for model_name, fitted in test_models.items():

    try:

        forecasts = np.asarray(
            fitted.forecast(steps=max_horizon)
        ).flatten()

        for horizon_name, steps in horizons.items():

            if steps > len(y_test):
                continue

            if steps > len(forecasts):
                continue

            actual = y_test.iloc[steps - 1]

            predicted = forecasts[steps - 1]

            actual_date = test_data.iloc[
                steps - 1
            ][date_col]

            if not np.isfinite(predicted):
                continue

            test_forecasts.append({

                "Dataset": "Testing",
                "Model": model_name,
                "Horizon": horizon_name,
                "Forecast_Origin": development_data[
                    date_col
                ].iloc[-1],
                "Actual_Target_Date": actual_date,
                "Actual": float(actual),
                "Predicted": float(predicted)

            })

    except Exception as error:
        print(f"{model_name} testing error:", error)

test_forecasts = pd.DataFrame(test_forecasts)

# ------------------------------------------------------------
# Evaluate forecasting performance
# ------------------------------------------------------------

def evaluate_forecasts(prediction_df):

    results = []

    if prediction_df.empty:
        return pd.DataFrame()

    for (model_name, horizon), group in prediction_df.groupby(
        ["Model", "Horizon"]
    ):

        actual = group["Actual"].values
        predicted = group["Predicted"].values

        mae = mean_absolute_error(actual, predicted)

        rmse = np.sqrt(
            mean_squared_error(actual, predicted)
        )

        nonzero = actual != 0

        mape = (
            mean_absolute_percentage_error(
                actual[nonzero],
                predicted[nonzero]
            ) * 100
            if nonzero.any()
            else np.nan
        )

        results.append({

            "Model": model_name,
            "Horizon": horizon,
            "Observations": len(group),
            "MAE": mae,
            "RMSE": rmse,
            "MAPE_Percent": mape,
            "Bias": np.mean(predicted - actual)

        })

    return pd.DataFrame(results)


validation_metrics = evaluate_forecasts(
    validation_forecasts
)

test_metrics = evaluate_forecasts(
    test_forecasts
)

# ------------------------------------------------------------
# Analyze model residuals
# ------------------------------------------------------------

residual_results = []
residual_series = {}

for model_name, fitted in validation_models.items():

    try:

        residuals = pd.Series(
            np.asarray(fitted.resid).flatten()
        ).replace(
            [np.inf, -np.inf],
            np.nan
        ).dropna()

        residual_series[model_name] = residuals

        residual_results.append({

            "Model": model_name,
            "Residual_Count": len(residuals),
            "Residual_Mean": residuals.mean(),
            "Residual_Std": residuals.std(),
            "Residual_Min": residuals.min(),
            "Residual_Max": residuals.max()

        })

    except Exception as error:
        print(f"Residual analysis skipped for {model_name}:", error)

residual_summary = pd.DataFrame(
    residual_results
)

# ------------------------------------------------------------
# Test residual autocorrelation
# ------------------------------------------------------------

ljung_box_results = []

for model_name, residuals in residual_series.items():

    if len(residuals) < 10:
        continue

    lag = min(10, max(1, len(residuals) // 5))

    try:

        lb_result = acorr_ljungbox(
            residuals,
            lags=[lag],
            return_df=True
        )

        p_value = lb_result["lb_pvalue"].iloc[0]

        ljung_box_results.append({

            "Model": model_name,
            "Lag": lag,
            "Ljung_Box_Statistic": lb_result[
                "lb_stat"
            ].iloc[0],
            "P_Value": p_value,
            "Residual_Autocorrelation": (
                "Present" if p_value < 0.05
                else "Not detected"
            )

        })

    except Exception as error:
        print(f"Ljung-Box test skipped for {model_name}:", error)

ljung_box_summary = pd.DataFrame(
    ljung_box_results
)

# ------------------------------------------------------------
# Compare statistical models
# ------------------------------------------------------------

statistical_comparison = (
    validation_metrics
    .sort_values(["Horizon", "MAE"])
    .reset_index(drop=True)
)

best_statistical_models = (
    statistical_comparison
    .groupby("Horizon")
    .first()
    .reset_index()
)

# ------------------------------------------------------------
# Save candidate parameter results
# ------------------------------------------------------------

arima_candidates_df.to_csv(
    os.path.join(OUTPUT_DIR, "arima_candidate_parameters.csv"),
    index=False
)

sarima_candidates_df.to_csv(
    os.path.join(OUTPUT_DIR, "sarima_candidate_parameters.csv"),
    index=False
)

# ------------------------------------------------------------
# Save forecast results
# ------------------------------------------------------------

validation_forecasts.to_csv(
    os.path.join(OUTPUT_DIR, "statistical_validation_forecasts.csv"),
    index=False
)

test_forecasts.to_csv(
    os.path.join(OUTPUT_DIR, "statistical_test_forecasts.csv"),
    index=False
)

validation_metrics.to_csv(
    os.path.join(OUTPUT_DIR, "statistical_validation_metrics.csv"),
    index=False
)

test_metrics.to_csv(
    os.path.join(OUTPUT_DIR, "statistical_test_metrics.csv"),
    index=False
)

statistical_comparison.to_csv(
    os.path.join(OUTPUT_DIR, "statistical_model_comparison.csv"),
    index=False
)

best_statistical_models.to_csv(
    os.path.join(OUTPUT_DIR, "best_statistical_models.csv"),
    index=False
)

# ------------------------------------------------------------
# Save residual diagnostics
# ------------------------------------------------------------

residual_summary.to_csv(
    os.path.join(OUTPUT_DIR, "statistical_residual_summary.csv"),
    index=False
)

ljung_box_summary.to_csv(
    os.path.join(OUTPUT_DIR, "statistical_ljung_box_results.csv"),
    index=False
)

residual_df = pd.DataFrame({
    name: values.reset_index(drop=True)
    for name, values in residual_series.items()
})

residual_df.to_csv(
    os.path.join(OUTPUT_DIR, "statistical_model_residuals.csv"),
    index=False
)

# ------------------------------------------------------------
# Save fitted model artifacts
# ------------------------------------------------------------

for model_name, fitted in validation_models.items():

    joblib.dump(
        fitted,
        os.path.join(
            MODEL_DIR,
            f"{model_name.lower()}_validation.joblib"
        )
    )

for model_name, fitted in test_models.items():

    joblib.dump(
        fitted,
        os.path.join(
            MODEL_DIR,
            f"{model_name.lower()}_final.joblib"
        )
    )

# ------------------------------------------------------------
# Save forecasting configuration
# ------------------------------------------------------------

statistical_config = {

    "target_variable": target_col,
    "date_column": date_col,
    "reporting_frequency": reporting_frequency,
    "seasonal_period": seasonal_period,
    "adf_statistic": adf_statistic,
    "adf_pvalue": adf_pvalue,
    "selected_differencing_order": differencing_order,
    "selected_arima_order": best_arima_order,
    "selected_sarima_order": best_sarima_order,
    "selected_seasonal_order": best_seasonal_order,
    "forecast_horizons": horizons,
    "training_observations": len(train_data),
    "validation_observations": len(validation_data),
    "testing_observations": len(test_data),
    "validation_forecast_origin": str(
        train_data[date_col].iloc[-1]
    ),
    "testing_forecast_origin": str(
        development_data[date_col].iloc[-1]
    ),
    "models": list(validation_models.keys()),
    "parameter_selection": "Training data AIC",
    "model_selection": "Validation MAE",
    "final_evaluation": "Held-out testing data"

}

with open(
    os.path.join(
        OUTPUT_DIR,
        "statistical_forecasting_configuration.json"
    ),
    "w"
) as file:

    json.dump(
        statistical_config,
        file,
        indent=4,
        default=str
    )

# ------------------------------------------------------------
# Final results
# ------------------------------------------------------------

print("\n============================================")
print("STATISTICAL FORECASTING COMPLETED")
print("============================================")

print("\nSelected ARIMA:", best_arima_order)
print("Selected SARIMA:", best_sarima_order)
print("Seasonal period:", seasonal_period)

print("\nTrained models:")

for model_name in validation_models:
    print("-", model_name)

print("\nVALIDATION PERFORMANCE")
print(validation_metrics.to_string(index=False))

print("\nTEST PERFORMANCE")
print(test_metrics.to_string(index=False))

print("\nBEST STATISTICAL MODELS")
print(best_statistical_models.to_string(index=False))

print("\nRESIDUAL AUTOCORRELATION")
print(ljung_box_summary.to_string(index=False))

print("\nModel artifacts:", MODEL_DIR)
print("Output directory:", OUTPUT_DIR)

print("\nAll statistical forecasting outputs saved.")


Dataset: C:\Users\hp\Predictive Forecasting of Care Load & Placement Demand\outputs\forecasting_framework_dataset.csv
Date column: date
Target: children_in_hhs_care
Observations: 690

Chronological split:
Training: 482
Validation: 104
Testing: 104

Forecast horizons: {'1_day': 1, '7_day': 7, '14_day': 14, '30_day': 30}

Stationarity test
ADF statistic: -1.8555957551406173
ADF p-value: 0.35322971375823625
Selected differencing: 1



Selected ARIMA order: (2, 1, 2)

Reporting frequency: Daily
Seasonal period: 7



Selected SARIMA order: (1, 1, 1)
Selected seasonal order: (1, 1, 1, 7)



Trained validation models:
- ARIMA
- SARIMA
- Simple_Exponential_Smoothing
- Holt_Linear_Trend
- Holt_Winters



STATISTICAL FORECASTING COMPLETED

Selected ARIMA: (2, 1, 2)
Selected SARIMA: (1, 1, 1)
Seasonal period: 7

Trained models:
- ARIMA
- SARIMA
- Simple_Exponential_Smoothing
- Holt_Linear_Trend
- Holt_Winters

VALIDATION PERFORMANCE
                       Model Horizon  Observations         MAE        RMSE  MAPE_Percent        Bias
                       ARIMA  14_day             1 1502.244744 1502.244744     31.773366 1502.244744
                       ARIMA   1_day             1  100.755794  100.755794      1.601332  100.755794
                       ARIMA  30_day             1 3424.165361 3424.165361    124.424613 3424.165361
                       ARIMA   7_day             1 1203.772176 1203.772176     23.640459 1203.772176
           Holt_Linear_Trend  14_day             1 1558.232057 1558.232057     32.957531 1558.232057
           Holt_Linear_Trend   1_day             1  104.799647  104.799647      1.665602  104.799647
           Holt_Linear_Trend  30_day             1 3398.14887